# 20 · Orchestrator plus specialists

## Goal

Connect a drafting specialist agent to the spine agent and define exactly
what crosses the handoff boundary — not the whole conversation, a
deliberate contract.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_copilot_client
settings = load_settings()
client = get_copilot_client(settings, delegated=True)


## Concept

Recall finding #3: there's no orchestration model to configure on the GHCP
harness. "Connected agents" is not a workaround for that — it's the actual
mechanism the harness gives you for decomposing work across agents, and it
composes with everything upstream: each connected agent has its own
`copilot.yaml`, its own instructions, its own (fixed, per finding #4) model.
The spine agent becomes an orchestrator not by gaining new orchestration
config but by gaining a connected agent it can hand off to.

**Handoff contract:** define explicitly what data crosses — supplier name,
the performance/spend summary, the requested tone — not "hand the specialist
the whole conversation and hope." A narrow, typed contract is what makes
`22`'s per-tier model swap safe later: the drafting specialist's contract
doesn't change when its underlying model does.


## Build


### The drafting specialist — its own workspace, its own model


In [ ]:
from csx.pac import copilot_init, copilot_push
from pathlib import Path

specialist = Path("../agents/drafting-specialist")
copilot_init(specialist)
(specialist / "instructions.md").write_text('''
# Drafting specialist

You draft renewal correspondence given a supplier name, a spend/performance
summary, and a requested tone. You do not look anything up yourself — you
only draft from what's handed to you. Never issue a deadline in a first
message. Never use threatening language.
''')
copilot_push(specialist)


### Wire the handoff contract into the spine agent


In [ ]:
import yaml
spine = Path("../agents/contract-renewal-desk")
copilot_yaml = yaml.safe_load((spine / "copilot.yaml").read_text())
copilot_yaml["connectedAgents"] = [{
    "id": "drafting-specialist",
    "schemaName": "crd_drafting-specialist",
    "description": "Hand off to this agent once you have a supplier name and a spend/performance summary ready, and the user wants correspondence drafted.",
    "contract": {
        "inputs": ["supplierName", "summary", "tone"],
        "outputs": ["draftText"],
    },
}]
(spine / "copilot.yaml").write_text(yaml.dump(copilot_yaml, sort_keys=False))

copilot_push(spine)
import subprocess
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_drafting-specialist"], check=True)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
suite = run_suite(client, cases=load_golden(tags=["multi-agent"]) + load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("20", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="second agent build+publish + handoff verification")


## Teardown


In [ ]:
print("No teardown — drafting-specialist persists; 21 adds a critic loop, 22 re-pins its model.")
